# CS106 Sudoku-Bench - Evaluation Notebook

Notebook này chỉ đọc các file JSON/log trong `cs106/outputs`, không gọi API. Mục tiêu là tổng hợp kết quả để đưa vào báo cáo/slide:

- solve rate theo model, mode và kích thước lưới;
- average correct placements cho multi-step;
- error type summary;
- thống kê thời gian gọi model nếu output có `time_seconds`;
- xuất CSV và biểu đồ vào `cs106/evaluation/eval_outputs`.

Sau khi có thêm kết quả 9x9 mới, chỉ cần move JSON xuống `cs106/outputs` rồi Run All lại notebook này.

In [2]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / "evaluation", Path.cwd() / "cs106" / "evaluation", Path.cwd() / "cs106"]:
    if (candidate / "evaluate_results.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

import pandas as pd
from IPython.display import Markdown, display

from evaluate_results import (
    build_summary_tables,
    collect_results,
    default_export_dir,
    export_tables,
    find_dataset_dir,
    find_outputs_dir,
    find_repo_root,
    plot_tables,
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1. Cấu hình thư mục

Mặc định notebook đọc `cs106/outputs`. Nếu muốn đọc thêm thư mục tạm, ví dụ `cs106/9x9/outputs`, thêm vào `EXTRA_OUTPUT_DIRS`.

In [3]:
REPO_ROOT = find_repo_root()
OUTPUT_DIRS = [find_outputs_dir(REPO_ROOT)]
DATASET_DIR = find_dataset_dir(REPO_ROOT)
EXPORT_DIR = default_export_dir(REPO_ROOT)

# Optional: bật dòng dưới nếu muốn đọc thêm output tạm trước khi move vào cs106/outputs.
EXTRA_OUTPUT_DIRS = [
    # REPO_ROOT / "cs106" / "9x9" / "outputs",
]
OUTPUT_DIRS.extend([p for p in EXTRA_OUTPUT_DIRS if Path(p).exists()])

display(Markdown(f"**Repo root:** `{REPO_ROOT}`"))
display(Markdown("**Output dirs:** " + ", ".join(f"`{p}`" for p in OUTPUT_DIRS)))
display(Markdown(f"**Dataset dir:** `{DATASET_DIR}`"))
display(Markdown(f"**Export dir:** `{EXPORT_DIR}`"))

**Repo root:** `D:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench`

**Output dirs:** `D:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\outputs`

**Dataset dir:** `D:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\dataset`

**Export dir:** `d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\evaluation\eval_outputs`

## 2. Load toàn bộ kết quả

Mỗi dòng là một run: một model, một puzzle, một mode.

In [4]:
df = collect_results(output_dirs=OUTPUT_DIRS, dataset_dir=DATASET_DIR)
tables = build_summary_tables(df)

display(Markdown(f"Loaded **{len(df)}** result files."))
display(df)

Loaded **56** result files.

,mode,grid_size,puzzle_id,difficulty,model,status,is_solved,error_type,correct_placements,placement_denominator,placement_accuracy,total_steps,correct_cells,total_cells,cell_accuracy,time_seconds,time_minutes,log_entries,relative_path,mismatch_preview
0,multi,6,1,easy,gemini-2.5-flash,Success,True,Success,36.0,36.0,1.000000,37.0,NaN,NaN,NaN,NaN,NaN,37,gemini-2.5/multi_prompt/flash/log_puzzle_01.json,
1,multi,6,2,easy,gemini-2.5-flash,Failed,False,Incorrect Solution,4.0,36.0,0.111111,5.0,NaN,NaN,NaN,NaN,NaN,5,gemini-2.5/multi_prompt/flash/log_puzzle_02.json,
2,multi,6,3,easy,gemini-2.5-flash,Success,True,Success,36.0,36.0,1.000000,36.0,NaN,NaN,NaN,NaN,NaN,36,gemini-2.5/multi_prompt/flash/log_puzzle_03.json,
3,multi,6,4,easy,gemini-2.5-flash,Success,True,Success,36.0,36.0,1.000000,36.0,NaN,NaN,NaN,NaN,NaN,36,gemini-2.5/multi_prompt/flash/log_puzzle_04.json,
4,multi,6,5,easy,gemini-2.5-flash,Success,True,Success,36.0,36.0,1.000000,36.0,NaN,NaN,NaN,NaN,NaN,36,gemini-2.5/multi_prompt/flash/log_puzzle_05.json,
7,multi,6,1,easy,gemini-2.5-pro,Success,True,Success,36.0,36.0,1.000000,36.0,NaN,NaN,NaN,NaN,NaN,36,gemini-2.5/multi_prompt/pro/log_puzzle_01.json,
8,multi,6,2,easy,gemini-2.5-pro,Success,True,Success,36.0,36.0,1.000000,36.0,NaN,NaN,NaN,NaN,NaN,36,gemini-2.5/multi_prompt/pro/log_puzzle_02.json,
9,multi,6,3,easy,gemini-2.5-pro,Success,True,Success,36.0,36.0,1.000000,36.0,NaN,NaN,NaN,NaN,NaN,36,gemini-2.5/multi_prompt/pro/log_puzzle_03.json,
10,multi,6,4,easy,gemini-2.5-pro,Success,True,Success,36.0,36.0,1.000000,37.0,NaN,NaN,NaN,NaN,NaN,37,gemini-2.5/multi_prompt/pro/log_puzzle_04.json,
11,multi,6,5,easy,gemini-2.5-pro,Failed,False,Incorrect Solution,4.0,36.0,0.111111,5.0,NaN,NaN,NaN,NaN,NaN,5,gemini-2.5/multi_prompt/pro/log_puzzle_05.json,


## 3. Solve rate theo model, mode và grid size

Đây là bảng chính để đưa vào slide/báo cáo. Paper gốc cũng báo cáo solve rate theo model, mode và kích thước puzzle.

In [5]:
summary = tables["summary_by_model_mode_size"].copy()
for col in ["solve_rate_pct", "avg_placement_accuracy_pct", "avg_time_seconds", "total_time_minutes", "avg_total_steps"]:
    if col in summary:
        summary[col] = summary[col].round(2)
display(summary)

TypeError: type NAType doesn't define __round__ method

## 4. Average correct placements cho multi-step

Metric này gần với phần paper gọi là **Multi-step correct placements**: trung bình model điền đúng được bao nhiêu ô trước khi solve hoặc fail.

In [ ]:
multi_correct = tables["multi_correct_placements"].copy()
for col in ["solve_rate_pct", "avg_correct_placements", "median_correct_placements", "max_correct_placements", "avg_steps"]:
    if col in multi_correct:
        multi_correct[col] = multi_correct[col].round(2)
display(multi_correct)

## 5. Chi tiết theo từng puzzle

Bảng này giúp rà nhanh puzzle nào model solve/fail, fail vì lỗi gì, đi được bao nhiêu bước.

In [ ]:
per_puzzle = tables["per_puzzle_status"].copy()
for col in ["time_seconds"]:
    if col in per_puzzle:
        per_puzzle[col] = per_puzzle[col].round(2)
display(per_puzzle)

## 6. Error type summary

Paper gốc có phân loại lỗi như Incorrect Solution, Surrender, Missing Information, Claimed Contradiction. Notebook này gom các lỗi hiện có trong log về các nhóm gần tương ứng, đồng thời giữ thêm `No Certain Move` cho multi-step.

In [ ]:
error_summary = tables["error_summary"].copy()
display(error_summary)

failed_only = error_summary[error_summary["error_type"] != "Success"]
display(Markdown("### Failed runs only"))
display(failed_only)

## 7. Thống kê thời gian

Một số output có `time_seconds`, đặc biệt single-prompt. Multi-step có thể thiếu thời gian tổng nếu notebook chạy không lưu field này.

In [ ]:
time_summary = tables["time_summary"].copy()
for col in ["avg_time_seconds", "median_time_seconds", "max_time_seconds", "total_time_minutes"]:
    if col in time_summary:
        time_summary[col] = time_summary[col].round(2)
display(time_summary)

display(Markdown("### Slowest runs"))
slowest = tables["slowest_runs"].copy()
for col in ["time_seconds", "time_minutes"]:
    if col in slowest:
        slowest[col] = slowest[col].round(2)
display(slowest[[c for c in ["grid_size", "mode", "puzzle_id", "model", "status", "time_seconds", "time_minutes", "relative_path"] if c in slowest.columns]])

## 8. Export CSV và biểu đồ

Các bảng CSV sẽ được lưu vào `cs106/evaluation/eval_outputs`. Nếu môi trường có `matplotlib`, notebook sẽ xuất thêm PNG chart.

In [ ]:
export_path = export_tables(tables, EXPORT_DIR)
chart_paths = plot_tables(tables, EXPORT_DIR)

display(Markdown(f"Exported CSV tables to: `{export_path}`"))
if chart_paths:
    display(Markdown("Saved charts:\n" + "\n".join(f"- `{p}`" for p in chart_paths)))
else:
    display(Markdown("No charts were created. Install/import `matplotlib` if you want PNG charts."))

## 9. Gợi ý đưa vào báo cáo

Các bảng nên đưa vào báo cáo/slide:

- `summary_by_model_mode_size.csv`: bảng solve rate chính;
- `multi_correct_placements.csv`: average correct placements cho multi-step;
- `error_summary.csv`: thống kê lỗi;
- `time_summary.csv`: thống kê thời gian;
- `per_puzzle_status.csv`: bảng phụ để kiểm tra chi tiết từng puzzle.

Khi có output 9x9 mới, chỉ cần move JSON vào `cs106/outputs` rồi Run All lại.